# CrewAI — agents with roles, tasks and crews

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhelamay/CPUT_ICAILY_AI2026_MaganLLM_WORKSHOP/blob/main/SA_LIB_WORKSHOP_Day2/02_crewai/02_crewai.ipynb)

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

CrewAI models a **team**: each Agent has a role, goal and backstory; Tasks say what to do; a Crew runs them. Great for multi-agent pipelines.

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "crewai>=1.0" "python-dotenv" "ddgs"

### API keys
* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

## 1. `01_first_agent.py`

CrewAI 01 — your first agent
CrewAI thinks in TEAMS: an Agent (who), a Task (what), and a Crew (runs the tasks).

In [ ]:
import asyncio
import os

os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")  # no cloud tracing prompt during the workshop

from crewai import LLM, Agent, Crew, Task

from workshop_common import get_llm_config

# the brain — "openai/<model>" + base_url means "any OpenAI-compatible provider"
cfg = get_llm_config()
llm = LLM(model=cfg.litellm_model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)

# the agent: CrewAI describes an agent with a role, a goal and a backstory (= the system prompt)
assistant = Agent(
    role="Helpful assistant",
    goal="Answer questions concisely and accurately",
    backstory="You explain technical ideas in plain language.",
    llm=llm,
)

# the task: what to do + what a good answer looks like
task = Task(
    description="explain what an AI agent is in two sentences.",
    expected_output="Two clear sentences.",
    agent=assistant,
)

async def main():
    # kickoff_async works both in scripts and in notebooks (which already run an event loop)
    result = await Crew(agents=[assistant], tasks=[task]).kickoff_async()
    print(f"[{cfg.provider}:{cfg.model}]", result.raw)


if __name__ == "__main__":
    await main()

## 2. `02_agent_with_tools.py`

CrewAI 02 — giving your agent tools
@tool("name") turns a python function into a CrewAI tool. the docstring tells the agent when to use it.

In [ ]:
import asyncio
import os

os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")

from crewai import LLM, Agent, Crew, Task
from crewai.tools import tool

import workshop_common as wc

cfg = wc.get_llm_config()
llm = LLM(model=cfg.litellm_model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)


# tool 1 — web search (free, no key)
@tool("web_search")
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


# tool 2 — a function you wrote yourself
@tool("word_count")
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


assistant = Agent(
    role="Research assistant",
    goal="Answer accurately, using tools when they help",
    backstory="You always check facts with a web search before answering.",
    tools=[web_search, word_count],
    llm=llm,
    verbose=True,   # prints every Thought / Action / Observation — the agent loop, live
)

task = Task(
    description="search for the latest langchain version, then tell me how many words your answer is.",
    expected_output="The latest version, plus the word count of your answer.",
    agent=assistant,
)

async def main():
    result = await Crew(agents=[assistant], tasks=[task]).kickoff_async()
    print(result.raw)


if __name__ == "__main__":
    await main()

## 3. `03_agent_with_memory.py`

CrewAI 03 — giving your agent memory
the simplest memory: keep the conversation in a list and send it every time.
(CrewAI also has Crew(memory=True) — long-term memory with a vector database — but it needs an
 embeddings provider, so for a free workshop we show the idea with a plain list.)

In [ ]:
import asyncio
import os

os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")

from crewai import LLM, Agent

from workshop_common import get_llm_config

cfg = get_llm_config()
llm = LLM(model=cfg.litellm_model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)

assistant = Agent(
    role="Helpful assistant",
    goal="Answer concisely and remember the conversation",
    backstory="You are friendly and precise.",
    llm=llm,
)

history = []  # <- the memory: same list = same conversation


async def chat(text: str) -> str:
    history.append({"role": "user", "content": text})
    reply = (await assistant.kickoff_async(history)).raw   # runs ONE agent without a crew
    history.append({"role": "assistant", "content": reply})
    return reply


async def main():
    print("\n>>> assistant:", await chat("hi! my name is m0h."))
    print("\n>>> assistant:", await chat("what's my name?"))


if __name__ == "__main__":
    await main()

## 4. `04_agent_with_fallback.py`

CrewAI 04 — the fallback setup
try each provider (LLM_PROVIDER, then LLM_FALLBACKS in .env) until one answers.

In [ ]:
import asyncio
import os

os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")

from crewai import LLM, Agent

from workshop_common import fallback_order, get_llm_config, ping


def get_llm() -> LLM:
    for provider in fallback_order():
        try:
            cfg = get_llm_config(provider)
            ping(cfg)
            print(f"using {provider} ({cfg.model})")
            return LLM(model=cfg.litellm_model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


assistant = Agent(role="Helpful assistant", goal="Answer concisely", backstory="You are precise.", llm=get_llm())
async def main():
    result = await assistant.kickoff_async("say hi and tell me which model you are.")
    print("\n>>> assistant:", result.raw)


if __name__ == "__main__":
    await main()

## 5. `05_multi_agent.py`

CrewAI 05 — multi-agent crew: researcher -> writer -> editor
this is what CrewAI is famous for. each agent has a role; each task's output
is automatically passed as CONTEXT to the next task (Process.sequential).

In [ ]:
import asyncio
import os

os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")

from crewai import LLM, Agent, Crew, Process, Task
from crewai.tools import tool

import workshop_common as wc

cfg = wc.get_llm_config()
llm = LLM(model=cfg.litellm_model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)


@tool("web_search")
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@tool("word_count")
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


researcher = Agent(role="Researcher", goal="Find accurate, current facts",
                   backstory="You search the web and report facts with sources.", tools=[web_search], llm=llm)
writer = Agent(role="Writer", goal="Turn facts into a short, friendly paragraph",
               backstory="You write for workshop attendees who are new to AI.", llm=llm)
editor = Agent(role="Editor", goal="Make sure the paragraph is under 60 words",
               backstory="You are strict about length. You use word_count to check.", tools=[word_count], llm=llm)

research = Task(description="Find the latest version of {topic} and one notable feature.",
                expected_output="3 bullet facts with sources.", agent=researcher)
write = Task(description="Write one paragraph about {topic} using the research.",
             expected_output="One paragraph.", agent=writer)
edit = Task(description="Check the paragraph with word_count and shorten it to under 60 words if needed.",
            expected_output="The final paragraph and its word count.", agent=editor)

crew = Crew(agents=[researcher, writer, editor], tasks=[research, write, edit], process=Process.sequential)
async def main():
    result = await crew.kickoff_async(inputs={"topic": "langchain"})  # fills {topic} in the tasks
    print(result.raw)


if __name__ == "__main__":
    await main()

## 6. `agent.py`

CrewAI — the complete agent (fallback + tools + memory)

In [ ]:
import asyncio
import os

os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")

from crewai import LLM, Agent
from crewai.tools import tool

import workshop_common as wc


def get_llm() -> LLM:
    for provider in wc.fallback_order():
        try:
            cfg = wc.get_llm_config(provider)
            wc.ping(cfg)
            print(f"using {provider} ({cfg.model})")
            return LLM(model=cfg.litellm_model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


@tool("web_search")
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@tool("word_count")
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


assistant = Agent(
    role="Helpful assistant",
    goal="Use your tools when they help answer accurately, and remember the conversation",
    backstory="You are friendly, precise and always check facts.",
    tools=[web_search, word_count],
    llm=get_llm(),
)
history = []


async def chat(text: str) -> str:
    history.append({"role": "user", "content": text})
    reply = (await assistant.kickoff_async(history)).raw
    history.append({"role": "assistant", "content": reply})
    return reply


async def main():
    print("\n>>> assistant:", await chat("my name is m0h. search for the latest langchain version, then tell me how many words your answer is."))
    print("\n>>> assistant:", await chat("what's my name?"))


if __name__ == "__main__":
    await main()

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*